# Analytics & Diversity

This notebook covers:
- Hill diversity numbers and curves
- PGEN distribution (analytical Gaussian mixture)
- Predicted richness and overlap (occupancy model)
- Sharing spectrum predictions
- Repertoire comparison (JSD, Jaccard)
- Perplexity and entropy metrics

In [ ]:
from LZGraphs import LZGraph, jensen_shannon_divergence, k_diversity, saturation_curve
import numpy as np
import csv

In [ ]:
# Load data and build graph
sequences, v_genes, j_genes = [], [], []
with open('ExampleData3.csv') as f:
    for row in csv.DictReader(f):
        sequences.append(row['cdr3_amino_acid'])
        v_genes.append(row['V'])
        j_genes.append(row['J'])

graph = LZGraph(sequences, variant='aap', v_genes=v_genes, j_genes=j_genes)
print(graph)

## 1. Hill Diversity Numbers

Hill numbers $D(\alpha)$ unify richness ($\alpha=0$), Shannon diversity ($\alpha=1$), and Simpson diversity ($\alpha=2$) into a single parametric family. Higher $\alpha$ gives more weight to dominant sequences.

All Hill numbers are computed analytically via LZ-constrained forward propagation — no simulation needed.

In [ ]:
# Individual Hill numbers
print(f'D(0) = {graph.hill_number(0):.1f}  (richness: number of distinct walks)')
print(f'D(1) = {graph.hill_number(1):.1f}  (exp Shannon entropy)')
print(f'D(2) = {graph.hill_number(2):.1f}  (inverse Simpson)')

In [ ]:
# Hill curve (diversity profile across orders)
curve = graph.hill_curve()
print('Order → D(order):')
for o, d in zip(curve['orders'][:8], curve['values'][:8]):
    print(f'  {o:6.2f} → {d:.2f}')

In [ ]:
# Effective diversity and uniformity
dp = graph.diversity_profile()
print(f'Effective diversity: {dp["effective_diversity"]:.2f}')
print(f'Entropy (nats):     {dp["entropy_nats"]:.4f}')
print(f'Entropy (bits):     {dp["entropy_bits"]:.4f}')
print(f'Uniformity:         {dp["uniformity"]:.4f}')

## 2. PGEN Distribution

The LZPGEN distribution describes the probability landscape of the graph — how generation probabilities are distributed across all possible sequences. LZGraphs computes this analytically as a Gaussian mixture model.

In [ ]:
# Moments of the log-PGEN distribution
m = graph.pgen_moments()
print(f'Mean log-PGEN: {m["mean"]:.4f}')
print(f'Std:           {m["std"]:.4f}')
print(f'Skewness:      {m["skewness"]:.4f}')
print(f'Kurtosis:      {m["kurtosis"]:.4f}')

In [ ]:
# Analytical Gaussian mixture distribution
dist = graph.pgen_distribution()
print(dist)
print(f'Components: {dist.n_components}')
print(f'Global mean: {dist.mean:.4f}')

In [ ]:
# PDF and CDF at specific points
x = np.linspace(m['mean'] - 3*m['std'], m['mean'] + 3*m['std'], 50)
pdf_vals = np.array([dist.pdf(xi) for xi in x])
cdf_vals = np.array([dist.cdf(xi) for xi in x])

print(f'PDF at mean: {dist.pdf(dist.mean):.4f}')
print(f'CDF at mean: {dist.cdf(dist.mean):.4f}')

In [ ]:
# Sample from the distribution
samples = dist.sample(10000, seed=42)
print(f'Sample mean:      {np.mean(samples):.4f} (analytical: {dist.mean:.4f})')
print(f'Sample std:       {np.std(samples):.4f} (analytical: {m["std"]:.4f})')

## 3. Occupancy Model — Predicted Richness & Overlap

Given the graph's probability distribution, predict how many **distinct** sequences you'd observe at a given sequencing depth $d$:

$$F(d) = \sum_s \left(1 - e^{-d \cdot \pi(s)}\right)$$

And the expected overlap between two samples at depths $d_i, d_j$.

In [ ]:
# Empirical richness from simulation (fast alternative to analytical occupancy)
from collections import Counter
result = graph.simulate(10000, seed=42)
seen = set()
depths_obs, richness_obs = [], []
for j, seq in enumerate(result.sequences, 1):
    seen.add(seq)
    if j in (100, 500, 1000, 5000, 10000):
        depths_obs.append(j)
        richness_obs.append(len(seen))

print('Depth → Unique sequences observed:')
for d, r in zip(depths_obs, richness_obs):
    print(f'  {d:>6,d} → {r:,d}')


In [ ]:
# Empirical overlap: simulate two independent samples
r1 = graph.simulate(5000, seed=1)
r2 = graph.simulate(5000, seed=2)
overlap = len(set(r1.sequences) & set(r2.sequences))
print(f'Shared sequences between two 5K samples: {overlap}')


In [ ]:
# Sharing spectrum (analytical — works best on small graphs)
small_g = LZGraph(sequences[:100], variant='aap')
sharing = small_g.predict_sharing([100, 100, 100])
print(f'Expected total unique across 3 donors: {sharing["expected_total"]:,.1f}')
print(f'Spectrum: {sharing["spectrum"][:4]}')


## 4. Repertoire Comparison

Compare two repertoires using Jensen-Shannon divergence and structural overlap.

In [ ]:
# Split into two sub-repertoires
g1 = LZGraph(sequences[:2500], variant='aap')
g2 = LZGraph(sequences[2500:], variant='aap')

jsd = jensen_shannon_divergence(g1, g2)
print(f'JSD(first half, second half) = {jsd:.6f}')

jsd_self = jensen_shannon_divergence(g1, g1)
print(f'JSD(self, self) = {jsd_self:.6f}')

In [ ]:
# Structural overlap via intersection
inter = g1 & g2
s1, s2, si = g1.summary(), g2.summary(), inter.summary()
total_nodes = s1['n_nodes'] + s2['n_nodes'] - si['n_nodes']
print(f'Jaccard (nodes): {si["n_nodes"] / total_nodes:.4f}')
print(f'Shared nodes: {si["n_nodes"]} / {s1["n_nodes"]} + {s2["n_nodes"]}')

## 5. Perplexity & Entropy

In [ ]:
# Perplexity of individual sequences
for seq in sequences[:5]:
    pp = graph.sequence_perplexity(seq)
    print(f'{seq:25s}  PP = {pp:.2f}')

In [ ]:
# Repertoire-level perplexity
pp = graph.repertoire_perplexity(sequences[:100])
print(f'Repertoire perplexity (100 seqs): {pp:.2f}')

In [ ]:
# Entropy rate
rate = graph.path_entropy_rate(sequences[:100])
print(f'Entropy rate: {rate:.4f} bits/token')

## 6. K-Diversity & Saturation

These are computed from raw sequences (no pre-built graph needed).

In [ ]:
kd = k_diversity(sequences, k=100, variant='aap', draws=50, seed=42)
print(f'K=100 diversity: {kd["mean"]:.1f} ± {kd["std"]:.1f}')
print(f'95% CI: [{kd["ci_low"]:.1f}, {kd["ci_high"]:.1f}]')

In [ ]:
# Saturation curve
points = saturation_curve(sequences, variant='aap', log_every=500)
print('Sequences → Nodes → Edges:')
for p in points:
    print(f'  {p["n_sequences"]:>5d}  →  {p["n_nodes"]:>5d}  →  {p["n_edges"]:>5d}')